In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path
import logging
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

In [2]:

# Загрузка датасета
DATASET_URL = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"
data = pd.read_csv(DATASET_URL)
print(f"Размер: {data.shape}")
data.head()


Размер: (6200, 48)


,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,...,nearest_public_transport_dist_m,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby
0,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,...,93.6,5,143.7,3,247.3,1,0.0,0.0,0,False
1,6.0,496.5,"HO CHI MIHN AVE, 19 ULYANOVSK","Россия, Ульяновск, проспект Хо Ши Мина, 19",48.300652,54.270443,Россия,Ульяновская область,городской округ Ульяновск,Ульяновск,...,89.9,6,NaN,0,260.8,1,0.0,220.4,2,False
2,7.0,496.5,SHELESTA 116A KHABAROVSK,"Россия, Хабаровск, улица Шелеста, 116А",135.052594,48.520497,Россия,Хабаровский край,городской округ Хабаровск,Хабаровск,...,33.8,8,112.9,6,0.0,0,0.0,186.6,2,False
3,8.0,496.5,ORDZHONIKIDZE 52 YAKUTSK,"Россия, Республика Саха (Якутия), Якутск, улиц...",129.721308,62.025566,Россия,Республика Саха (Якутия),городской округ Якутск,Якутск,...,119.8,7,246.9,4,195.4,5,0.0,167.2,1,False
4,10.0,496.5,"VETERANOV AVE, 3 KRASNOKAMENS","Россия, Забайкальский край, Краснокаменск, про...",118.027480,50.090714,Россия,Забайкальский край,Краснокаменский муниципальный округ,Краснокаменск,...,70.6,3,48.3,5,NaN,0,NaN,NaN,0,False


## EDA 

In [3]:
# Конфигурация на основе датасета
EXCLUDE_COLS = [
    "id", "address_raw", "address_geocoded", "street", "house", 
    "geo_lon", "geo_lat", "municipality", "country", "atm_group" # ПОКА ВЫКИНУЛА atm_group из обучения 
]

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement", 
    "access_for_disabled", "transfer_p2p", "transfer_a2a", "loan_payments", 
    "has_subway_nearby"
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m", "nearest_restaurants_dist_m",
    "count_restaurants_300m", "nearest_public_transport_dist_m",
    "count_public_transport_300m", "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m", "nearest_subway_dist_m",
    "nearest_post_offices_dist_m"
]

CATEGORICAL_FEATURES = ["city"]
TARGET = "target"

print(f"Бинарных фичей: {len(BIN_FEATURES)}")
print(f"Локационных фичей: {len(LOCATION_FEATURES)}")


Бинарных фичей: 14
Локационных фичей: 19


## INPUT DATASET vs STANDART COLUMNS FOR PYPLINE

In [4]:
class RegionGrouper(BaseEstimator, TransformerMixin):
   
    def __init__(self, rare_threshold=0.01):
        self.rare_threshold = rare_threshold
        self.rare_regions_ = None
    
    def fit(self, X, y=None):
        df = pd.DataFrame(X)
        if 'region' in df.columns:
            cnt = df['region'].value_counts()
            self.rare_regions_ = set(cnt[cnt < self.rare_threshold * len(df)].index)
        return self
    
    def transform(self, X):
        df = pd.DataFrame(X)
        if 'region' in df.columns and self.rare_regions_ is not None:
            df['region_grouped'] = df['region'].apply(
                lambda x: '__OTHER__' if pd.isna(x) or x in self.rare_regions_ else str(x)
            )
            df.drop(columns=['region'], inplace=True, errors='ignore')
        return df

class BinaryToFloat(BaseEstimator, TransformerMixin):
    def transform(self, X):
        df = pd.DataFrame(X)
        available_bin = [col for col in BIN_FEATURES if col in df.columns]
        for col in available_bin:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)
        return df
    def fit(self, X, y=None): return self

class ATMLocationFeatures(BaseEstimator, TransformerMixin):
    def transform(self, X):
        df = pd.DataFrame(X)
        dist_cols = [col for col in df.columns if 'dist_m' in col]
        for col in dist_cols:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))
        count_cols = [col for col in df.columns if 'count_' in col]
        if count_cols:
            df['poi_density_300m'] = df[count_cols].sum(axis=1)
        return df
    def fit(self, X, y=None): return self


In [5]:


# Доступные колонки из наших списков
available_location = [col for col in LOCATION_FEATURES if col in data.columns]
available_bin = [col for col in BIN_FEATURES if col in data.columns]
available_cat = [col for col in CATEGORICAL_FEATURES if col in data.columns]

print(f"\nДоступно:")
print(f"   Локационные: {len(available_location)}/{len(LOCATION_FEATURES)}")
print(f"   Бинарные:     {len(available_bin)}/{len(BIN_FEATURES)}")
print(f"   Категории:    {len(available_cat)}/{len(CATEGORICAL_FEATURES)}")

# Безопасная подготовка X
exclude_cols_present = [col for col in EXCLUDE_COLS if col in data.columns]
X = data.drop(columns=exclude_cols_present + [TARGET], errors='ignore')





Доступно:
   Локационные: 19/19
   Бинарные:     14/14
   Категории:    1/1


## Preprocessor


In [6]:
def create_preprocessor(X):

    
    all_cols = X.columns.tolist()
    available_num = [col for col in LOCATION_FEATURES if col in all_cols]
    available_bin = [col for col in BIN_FEATURES if col in all_cols]
    
    print(f"НАЙДЕНЫ (БЕЗ atm_group):")
    print(f"num: {len(available_num)}")
    print(f"bin: {len(available_bin)}")
    

    initial_pipeline = Pipeline([
        ('region_grouper', RegionGrouper()),  # region → region_grouped
        ('binary_to_float', BinaryToFloat()),
        ('location_features', ATMLocationFeatures())
    ])
    
  
    transformers = [
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), available_num),
        ('bin', SimpleImputer(strategy='most_frequent'), available_bin),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ]), CATEGORICAL_FEATURES)  # Только ['city']
    ]
    
    preprocessor = ColumnTransformer(transformers, remainder='drop')
    

    full_pipeline = Pipeline([
        ('initial', initial_pipeline),
        ('preprocessor', preprocessor),
        ('final_scaler', StandardScaler())
    ])
    
    return full_pipeline

full_pipeline = create_preprocessor(X)


НАЙДЕНЫ (БЕЗ atm_group):
num: 19
bin: 14


## Pypline Preprocessor Validate and Safe

In [7]:

full_pipeline.fit(X)

print("Тест:")
X_processed = full_pipeline.transform(X[:100])
print(f"Shape: {X_processed.shape}")
print(f"NA: {np.isnan(X_processed).sum()}")
print(f"Finite: {np.isfinite(X_processed).all()}")


import os
from pathlib import Path

MODEL_DIR = Path("/Users/shon/Downloads/ИИ пространство/Групповой проект")
MODEL_DIR.mkdir(parents=True, exist_ok=True)  # Создаем если нет

save_path = MODEL_DIR / "preprocessor.pkl"

try:
    joblib.dump(full_pipeline, save_path)
    print(f"✅ СОХРАНЕНО: {save_path}")
    print(f"Папка: {MODEL_DIR}")
    print(f"Размер файла: {save_path.stat().st_size / 1024:.1f} KB")
except Exception as e:
    print(f"❌ Ошибка: {e}")


Тест:
Shape: (100, 802)
NA: 0
Finite: True
✅ СОХРАНЕНО: /Users/shon/Downloads/ИИ пространство/Групповой проект/preprocessor.pkl
Папка: /Users/shon/Downloads/ИИ пространство/Групповой проект
Размер файла: 48.6 KB


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [8]:
# === Разделение данных на признаки и целевую переменную ===
# === Удаляем строки с пропущенными значениями ===
# data = data.dropna()

# === Удаляем координаты ===
X = data.drop(columns=['target', 'geo_lon', 'geo_lat'])
y = data['target']

# === Кодируем категориальный признак region ===
X = pd.get_dummies(X, columns=['region'], drop_first=True)

# === Теперь все признаки числовые ===
X = X.select_dtypes(include=['int64', 'float64', 'uint8'])

# === Базовый сплит данных (25% тест, seed = 42) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


## Train

In [9]:
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# === Разделение данных на признаки и целевую переменную ===
# === Удаляем строки с пропущенными значениями ===
# === Удаляем координаты ===
X = data.drop(columns=['target'])
y = data['target']

# === Базовый сплит данных (25% тест, seed = 42) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Загружаем обученный preprocessor
MODEL_DIR = Path("/Users/shon/Downloads/ИИ пространство/Групповой проект")
preprocessor = joblib.load(MODEL_DIR / "preprocessor.pkl")

print(f"X shape: {X_train.shape}")




X shape: (4650, 47)


In [11]:
from sklearn.pipeline import Pipeline

models = {
    'LinearRegression': LinearRegression(),
    'LassoCV': LassoCV(cv=5, random_state=42),
    'RidgeCV': RidgeCV(cv=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42, max_depth=10)
}

results = {}

for name, base_model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),  # работает по сырым X
        ('model', base_model),
    ])
    
    cv_scores = cross_val_score(
        pipe,
        X_train,  # СЫРЫЕ признаки, без .transform
        y_train,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    
    results[name] = {
        'CV_RMSE_mean': -cv_scores.mean(),
        'CV_RMSE_std': cv_scores.std(),
        'CV_scores': cv_scores
    }
    

/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in c

In [12]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

# Таблица результатов
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('CV_RMSE_mean')
results_df.rename(columns={
    'CV_RMSE_mean': 'rmse_mean',
    'CV_RMSE_std': 'rmse_std'
}, inplace=True)
display(results_df)

best_model_name = results_df.index[0]
print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")

best_base_model = models[best_model_name]
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_base_model),
])

# Обучаем на train
best_pipeline.fit(X_train, y_train)

# Оценка на test
y_pred_test = best_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(mse)

print(f"\n{best_model_name}")
print(f"   CV RMSE (mean): {results_df.loc[best_model_name, 'rmse_mean']:.4f}")
print(f"   CV RMSE (std):  {results_df.loc[best_model_name, 'rmse_std']:.4f}")
print(f"   Test RMSE:      {test_rmse:.4f}")


,rmse_mean,rmse_std,CV_scores
LassoCV,0.067343,0.001136,"[-0.0694765466341922, -0.06618560807355559, -0..."
RidgeCV,0.068855,0.001319,"[-0.07139369408656046, -0.06785311780824345, -..."
DecisionTree,0.069624,0.002529,"[-0.07374320483498667, -0.06837773525934059, -..."
LinearRegression,0.086183,0.032952,"[-0.07257390574300196, -0.15201825169843347, -..."


ЛУЧШАЯ МОДЕЛЬ: LassoCV

LassoCV
   CV RMSE (mean): 0.0673
   CV RMSE (std):  0.0011
   Test RMSE:      0.0705


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [13]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

test_results = {}

for name, base_model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', base_model),
    ])
    
    # Обучаем на train
    pipe.fit(X_train, y_train)
    
    # Предсказываем на test
    y_pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    test_results[name] = {
        'test_rmse': rmse
    }
    print(f"{name:15} | Test RMSE: {rmse:.4f}")

test_results_df = pd.DataFrame(test_results).T.sort_values('test_rmse')
display(test_results_df)


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


LinearRegression | Test RMSE: 0.0715


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


LassoCV         | Test RMSE: 0.0705


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


RidgeCV         | Test RMSE: 0.0712
DecisionTree    | Test RMSE: 0.0684


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,test_rmse
DecisionTree,0.068410
LassoCV,0.070531
RidgeCV,0.071160
LinearRegression,0.071503


In [14]:
# ===============================
# FINAL ATM PIPELINE (ONE BLOCK)
# ===============================

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

# ===============================
# CONFIG
# ===============================

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement",
    "access_for_disabled", "transfer_p2p", "transfer_a2a",
    "loan_payments", "has_subway_nearby"
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m",
    "nearest_restaurants_dist_m", "count_restaurants_300m",
    "nearest_public_transport_dist_m", "count_public_transport_300m",
    "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m",
    "nearest_subway_dist_m", "nearest_post_offices_dist_m"
]

CATEGORICAL_FEATURES = ["city"]

TARGET = "target"

MODEL_DIR = Path("/Users/shon/Downloads/ИИ пространство/Групповой проект")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "final_atm_pipeline.pkl"

# ===============================
# CUSTOM TRANSFORMERS
# ===============================

class RegionGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, rare_threshold=0.01):
        self.rare_threshold = rare_threshold
        self.rare_regions_ = None

    def fit(self, X, y=None):
        df = pd.DataFrame(X)
        if "region" in df.columns:
            cnt = df["region"].value_counts()
            self.rare_regions_ = set(cnt[cnt < self.rare_threshold * len(df)].index)
        return self

    def transform(self, X):
        df = pd.DataFrame(X)
        if "region" in df.columns and self.rare_regions_ is not None:
            df["region_grouped"] = df["region"].apply(
                lambda x: "__OTHER__" if pd.isna(x) or x in self.rare_regions_ else str(x)
            )
            df.drop(columns=["region"], inplace=True)
        return df


class BinaryToFloat(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = pd.DataFrame(X)
        for col in BIN_FEATURES:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)
        return df


class ATMLocationFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = pd.DataFrame(X)

        dist_cols = [c for c in df.columns if "dist_m" in c]
        for col in dist_cols:
            df[f"{col}_log"] = np.log1p(df[col].fillna(0))

        count_cols = [c for c in df.columns if c.startswith("count_")]
        if count_cols:
            df["poi_density_300m"] = df[count_cols].sum(axis=1)

        return df


# ===============================
# PREPROCESSOR
# ===============================

def build_preprocessor(X: pd.DataFrame) -> Pipeline:
    available_num = [c for c in LOCATION_FEATURES if c in X.columns]
    available_bin = [c for c in BIN_FEATURES if c in X.columns]

    column_transformer = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), available_num),

            ("bin", SimpleImputer(strategy="most_frequent"), available_bin),

            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(
                    drop="first",
                    sparse_output=False,
                    handle_unknown="ignore"
                ))
            ]), CATEGORICAL_FEATURES),
        ],
        remainder="drop"
    )

    return Pipeline([
        ("region_grouper", RegionGrouper()),
        ("binary_cast", BinaryToFloat()),
        ("location_features", ATMLocationFeatures()),
        ("columns", column_transformer),
        ("scaler", StandardScaler()),
    ])


# ===============================
# FINAL PIPELINE
# ===============================

preprocessor = build_preprocessor(X)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_base_model),   # LassoCV — лучшая модель
])

# ===============================
# TRAIN ON FULL DATA
# ===============================

final_pipeline.fit(X, y)

y_pred = final_pipeline.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE (FULL DATA): {rmse:.4f}")

# ===============================
# SAVE
# ===============================

joblib.dump(final_pipeline, MODEL_PATH)
print(f"FINAL PIPELINE SAVED TO: {MODEL_PATH}")


RMSE (FULL DATA): 0.0634
FINAL PIPELINE SAVED TO: /Users/shon/Downloads/ИИ пространство/Групповой проект/final_atm_pipeline.pkl


In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # поднимаемся из nb/ в корень
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("app exists:", (PROJECT_ROOT / "app").exists())


Project root: /Users/shon/Downloads/ИИ пространство/Групповой проект/Test model
app exists: True


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np
import joblib
from pathlib import Path

best_base_model = models[best_model_name]

from app.models.transformers import (
    RegionGrouper,
    BinaryToFloat,
    ATMLocationFeatures,
    DropNonNumeric,
)

final_pipeline = Pipeline([
    ("region_grouper", RegionGrouper()),
    ("binary_cast", BinaryToFloat()),
    ("location_features", ATMLocationFeatures()),
    ("drop_non_numeric", DropNonNumeric()),
    
    # ⬇⬇⬇ ВАЖНОЕ МЕСТО
    ("imputer", SimpleImputer(strategy="median")),
    
    ("model", best_base_model),  # LassoCV
])

# обучение
final_pipeline.fit(X, y)

# in-sample проверка
y_pred = final_pipeline.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE (FULL DATA): {rmse:.4f}")

MODEL_PATH = Path("app/models/final_atm_pipeline.pkl")

# ⬅⬅⬅ ВАЖНО: создать директорию
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_pipeline, MODEL_PATH)

print(f"PIPELINE SAVED TO: {MODEL_PATH.resolve()}")


RMSE (FULL DATA): 0.0633
PIPELINE SAVED TO: /Users/shon/Downloads/ИИ пространство/Групповой проект/Test model/nb/app/models/final_atm_pipeline.pkl


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np
import joblib
from pathlib import Path

from app.models.transformers import (
    RegionGrouper,
    BinaryToFloat,
    ATMLocationFeatures,
)

best_base_model = models[best_model_name]

final_pipeline = Pipeline([
    ("region_grouper", RegionGrouper()),
    ("binary_to_float", BinaryToFloat()),
    ("location_features", ATMLocationFeatures()),
    ("model", best_base_model),
])

# обучение
final_pipeline.fit(X, y)

# in-sample проверка
y_pred_all = final_pipeline.predict(X)
rmse_all = np.sqrt(mean_squared_error(y, y_pred_all))
print(f"RMSE на FULL датасете: {rmse_all:.4f}")

# сохранение
MODEL_DIR = Path("app/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

final_path = MODEL_DIR / "final_atm_pipeline.pkl"
joblib.dump(final_pipeline, final_path)

print(f"ФИНАЛЬНЫЙ ПАЙПЛАЙН СОХРАНЁН: {final_path}")


ValueError: could not convert string to float: 'BUDENNOGO 7A              ELISTA      '

In [ ]:
# test_model_load.py 
model = joblib.load("УКАЗАТЬ ПУТЬ К МЕСТУУ ХРАНЕНИЯ ПАЙПЛАЙНА")
print("Steps:", model.named_steps.keys())
print("Model type:", type(model.named_steps["model"]))

sample = pd.DataFrame([{
    "id": 5.0,
    "atm_group": 496.5,
    "address_raw": "BUDENNOGO 7A              ELISTA      ",
    "address_geocoded": "Россия, Республика Калмыкия, Элиста, улица С.М. Будённого, 7А",
    "geo_lon": 44.260605,
    "geo_lat": 46.318231,
    "country": "Россия",
    "region": "Республика Калмыкия",
    "municipality": "городской округ Элиста",
    "city": "Элиста",
    "street": "улица С.М. Будённого",
    "house": "7А",
    "population_density_per_km2": 3.52,
    "is_24_7": False,
    "contactless_tech": False,
    "qr_codes": False,
    "usd_available": False,
    "eur_available": False,
    "cash_in": True,
    "cash_out": True,
    "cashless_pay": False,
    "account_statement": True,
    "access_for_disabled": True,
    "transfer_p2p": True,
    "transfer_a2a": False,
    "loan_payments": False,
    "nearest_malls_dist_m": 928.7,
    "count_malls_300m": 0,
    "nearest_supermarkets_dist_m": 364.3,
    "count_supermarkets_300m": 0,
    "nearest_pharmacies_hospitals_dist_m": 242.6,
    "count_pharmacies_hospitals_300m": 2,
    "count_banks_atms_300m": 2,
    "nearest_cafes_dist_m": 124.5,
    "count_cafes_300m": 1,
    "nearest_restaurants_dist_m": 317.9,
    "count_restaurants_300m": 0,
    "nearest_public_transport_dist_m": 93.6,
    "count_public_transport_300m": 5,
    "nearest_parking_dist_m": 143.7,
    "count_parking_300m": 3,
    "nearest_education_dist_m": 247.3,
    "count_education_300m": 1,
    "nearest_subway_dist_m": 0.0,
    "nearest_post_offices_dist_m": 0.0,
    "count_post_offices_300m": 0,
    "has_subway_nearby": False,
}])

pred = model.predict(sample)
print(pred[0])


FileNotFoundError: [Errno 2] No such file or directory: 'УКАЗАТЬ ПУТЬ К МЕСТУУ ХРАНЕНИЯ ПАЙПЛАЙНА'

In [ ]:
# app/core/model.py
from pathlib import Path
from typing import Dict, Any
import joblib
import pandas as pd

MODEL_PATH = Path("УКАЗАТЬ ПУТЬ К МЕСТУУ ХРАНЕНИЯ ПАЙПЛАЙНА")

class ATMModelService:
    def __init__(self, model_path: Path = MODEL_PATH):
        self.model_path = model_path
        self.model = self._load_model()

    def _load_model(self):
        if not self.model_path.exists():
            raise FileNotFoundError(f"Model not found: {self.model_path}")
        return joblib.load(self.model_path)

    def predict_popularity(self, features: Dict[str, Any]) -> float:
        df = pd.DataFrame([features])
        y_pred = self.model.predict(df)[0]
        return float(y_pred)

# singleton для использования в FastAPI
atm_model_service = ATMModelService()
